In [30]:
from src.gaussian_mixture import GaussianMixtureMAR
from src.motivating_example import MotivatingExample
from src.data_generator import DataGenerator
import numpy as np

In [31]:
import numpy as np
import torch
import rpy2.robjects as ro
from rpy2.robjects.packages import importr

mice = importr("mice")


class AmputeMAR:
    
    def _to_r_dataframe(self, X):
        return ro.DataFrame({
            f"X{j + 1}": ro.FloatVector(X[:, j])
            for j in range(X.shape[1])
        })

    def _to_r_matrix(self, X):
        return ro.r.matrix(
            ro.IntVector(X.T.flatten().tolist()),
            nrow=X.shape[0],
            ncol=X.shape[1],
        )

    def ampute(self, X, patterns=None, prop=0.3):
        """
        Parameters
        ----------
        X : torch.Tensor, shape (n, d)
            Complete data.
        patterns : array-like, shape (n, d)
            Binary patterns: 1 = observed, 0 = missing.
            All-ones rows are kept complete.
        prop : float
            Proportion of eligible cases to ampute.

        Returns
        -------
        X_NA : torch.Tensor
            Data with NaN values.
        M : torch.Tensor
            Observation mask: 1 = observed, 0 = missing.
        """
        if patterns is None:
            patterns = np.ones_like(X, dtype=int)
            n, d = X.shape
            patterns[:, :-1] = np.random.randint(0, 2, size=(n, d - 1))
            patterns[:, -1] = 1

        device = X.device
        X_np = X.detach().cpu().numpy().astype(float)
        patterns = np.asarray(patterns)


        result = mice.ampute(
            self._to_r_dataframe(X),
            patterns=self._to_r_matrix(patterns),
            prop=prop,
            mech="MAR",
            bycases=True,
        )

        X_NA_np = X_np.copy()
        X_NA_np = np.asarray(result.rx2("amp")).T

        M_np = (~np.isnan(X_NA_np)).astype(np.float32)

        return (
            torch.tensor(X_NA_np, dtype=X.dtype, device=device),
            torch.tensor(M_np, dtype=X.dtype, device=device),
        )

In [32]:
class ParametricEstimation(GaussianMixtureMAR, AmputeMAR):

    def __init__(self, 
                 data_generator,
                 n,
                 d,
                 patterns,
                 k_range, 
                 criterion="bic", 
                 cov_type='uni_diag', 
                 max_iter=100, 
                 tol=0.000001, 
                 eps=0.000001, 
                 reg_covar=0.000001, 
                 n_init=20, 
                 _k_tol=1, 
                 device="cpu", 
                 verbose=False):
        super().__init__(k_range, criterion, cov_type, max_iter, tol, eps, reg_covar, n_init, _k_tol, device, verbose)
        self.n = n
        self.d = d
        self.X = data_generator.sample_truth(n, d)
        self.Xna, self.M = self.ampute(self.X, patterns=patterns)
        self.model = self.fit(self.X, self.M)

    def mean(self):
        Xtilde = self.model.sample(self.n)[0]
        return Xtilde.mean(dim=0)

    def quantile(self, var_index=0, p=0.1):
        """
        Estimate the p-th quantile of the variable with index var_index
        """
        Xtilde = self.model.sample(self.n)[0]
        x = Xtilde[: var_index]
        return np.quantile(x, p)

    def estimate(self, func):
        Xtilde = self.model.sample(self.n)[0]
        return func(Xtilde)

    # def truth(self, func):
    #     """the truth not from here, need to know data generating mechanism"""
    #     pass

In [33]:
dg = DataGenerator(distr='Logistic')

In [34]:
n = 2000
d = 3
B = 2

In [35]:
means = torch.empty((B, d))
quantiles_X2 = torch.empty(B)

for i in range(2):
    pe = ParametricEstimation(data_generator=dg, n=n, d=d, patterns=None, k_range=(0,10))
    means[i] = pe.mean()
    quantiles_X2[i] = pe.quantile(var_index=1, p=0.2).item()

In [36]:
print('Mean estimates per batch: \n', means)

Mean estimates per batch: 
 tensor([[-0.0220, -0.1234,  0.0538],
        [-0.0686, -0.1239,  0.0352]])


In [37]:
print('Quantile 0.2 estimates of X2 per batch: \n', quantiles_X2)

Quantile 0.2 estimates of X2 per batch: 
 tensor([-0.6150, -0.8516])
